In [32]:

from langchain_openai import OpenAIEmbeddings
import os

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='Qwen/Qwen3-32B',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)
embedding = OpenAIEmbeddings(
    model='Qwen/Qwen3-Embedding-8B',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)
os.environ['http_proxy'] = 'http://127.0.0.1:10809'
os.environ['https_proxy'] = 'http://127.0.0.1:10809'

In [33]:
persist_dir = 'chroma_youtube_data'
# 一些YouTube的视频连接
urls = [
    "https://www.youtube.com/watch?v=HAn9vnJy6S4",
    "https://www.youtube.com/watch?v=dA1cHGACXCo",
    "https://www.youtube.com/watch?v=ZcEMLz27sL4",
    "https://www.youtube.com/watch?v=hvAPnpSfSGo",
    "https://www.youtube.com/watch?v=EhlPDL4QrWY",
    "https://www.youtube.com/watch?v=mmBo8nlu2j0",
    "https://www.youtube.com/watch?v=rQdibOsL1ps",
    "https://www.youtube.com/watch?v=28lC4fqukoc",
    "https://www.youtube.com/watch?v=es-9MgxB-uc",
    "https://www.youtube.com/watch?v=wLRHwKuKvOE",
    "https://www.youtube.com/watch?v=ObIltMaRJvY",
    "https://www.youtube.com/watch?v=DjuXACWYkkU",
    "https://www.youtube.com/watch?v=o7C9ld6Ln-M",
]
docs = []  # document的数组
# YoutubeLoader会发生400 BadRequest 暂时不处理
# 社区的遗留问题 目前没搜到解决方案
# from langchain_community.document_loaders import YoutubeLoader
#
# for url in urls:
#     docs.extend(YoutubeLoader.from_youtube_url(url, add_video_info=True).load())
# len(docs)

In [34]:
# YoutubeLoader会发生400 BadRequest 暂时不处理
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
#
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=30)
# split_doc = text_splitter.split_documents(docs)
# # 加载到Chroma向量数据库
# vector_store = Chroma.from_documents(split_doc, embedding, persist_directory=persist_dir)

In [35]:
from langchain_core.documents import Document
# 从gpt的向量数据中重建qwen的向量数据库
# vector_store_gpt = Chroma(persist_directory=persist_dir, embedding_function=embedding)
#
#
# old_data = vector_store_gpt.get()
# documents = old_data['documents']
# metadatas = old_data['metadatas']
# ids = old_data['ids']
# doc_objects = [
#     Document(page_content=doc, metadata=meta)
#     for doc, meta in zip(documents, metadatas)
# ]
# vector_store_qwen = Chroma.from_documents(
#     documents=doc_objects,
#     embedding=embedding,
#     persist_directory=f'{persist_dir}_qwen'
# )
# 加载前问的数据库
vector_store_qwen = Chroma(persist_directory=f'{persist_dir}_qwen', embedding_function=embedding)

In [36]:
vector_store_qwen.similarity_search('I like RAG')

[Document(id='e7c523d3-0f97-4933-8947-1e51c2ca3cf5', metadata={'view_count': 2676, 'publish_date': '2024-01-24 00:00:00', 'source': 'ZcEMLz27sL4', 'thumbnail_url': 'https://i.ytimg.com/vi/ZcEMLz27sL4/hq720.jpg?sqp=-oaymwEmCIAKENAF8quKqQMa8AEB-AHUCIAC0AWKAgwIABABGGQgZChkMA8=&rs=AOn4CLBZCZLx203dWs9DKzqrJY3Lvxpu6A', 'length': 1272, 'title': 'Streaming Events: Introducing a new `stream_events` method', 'description': 'Unknown', 'author': 'LangChain', 'publish_year': 2024}, page_content="we had before so we have the input we'll use version one we're using a stream events and now we can start doing stuff with the uh the events that are emitted so if onchain start and if the event name was agent so this is basically this is if you remember back up here we tagged the agent executor with the Run name agent so this is basically saying whenever on on the start and end of this we are going to print out this and then same thing on the end of this we're going to print out this and this is needed bec

In [37]:
from langchain_core.prompts import ChatPromptTemplate

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a list of database queries optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)


In [38]:
from pydantic import BaseModel, Field
from typing import Optional


# pydantic 模型: 描述数据结构
class Search(BaseModel):
    # 内容的相似性和发布年份
    query: str = Field(None, description='Similarity search query applied to video transcripts.')
    publish_year: Optional[int] = Field(None, description='Year video was published')

In [39]:
from langchain_core.runnables import RunnablePassthrough

# 将用户输入的内容转化为搜索条件
chain = {'question': RunnablePassthrough()} | prompt | model.with_structured_output(Search)

In [40]:
chain.invoke('how do I build a RAG agent?')

Search(query='how do I build a RAG agent?', publish_year=2024)

In [41]:
chain.invoke('videos on RAG published in 2023')

Search(query='RAG', publish_year=2023)

In [42]:
from typing import List


# 带条件的向量数据库检索
def retrieval(search: Search) -> List[Document]:
    _filters = None
    if search.publish_year:
        _filters = {'publish_year': {"$eq": search.publish_year}}
    return vector_store_qwen.similarity_search(search.query, filter=_filters)

In [43]:
chain = chain | retrieval

In [45]:
resp = chain.invoke('videos on RAG published in 2023')